# Concrete QC project - Step 1b (data cleaning)

Goal: decide which rows we can trust and get the data ready for Step 1c.

We do NOT delete anything from the raw files. Instead every cube result gets a column called `remove_reason`.
If it is empty, the row is used. If it has a reason, the row is left out of the main analysis.
This way we can always see what we removed and why.

At the end we save:
- `data/interim/cubes_with_reasons.csv` - all rows with the reason column
- `data/processed/cubes_clean.csv` - only the rows we keep
- `data/processed/durability_clean.csv`
- `data/processed/mix_designs.csv`
- `reports/cleaning_log.csv` - how many rows each rule removed

### Step 1: Load the files

In [1]:
from pathlib import Path
import pandas as pd

project = Path.cwd().parent   # the notebook runs from the notebooks folder, so the project is one level up
raw = project / "data" / "raw"

cubes = pd.read_csv(raw / "cube_results.csv")
mixes = pd.read_csv(raw / "mix_designs.csv")
durability = pd.read_csv(raw / "durability_results.csv")

print("Cubes:", cubes.shape)
print("Mixes:", mixes.shape)
print("Durability:", durability.shape)

Cubes: (4079, 31)
Mixes: (12, 16)
Durability: (198, 29)


## PART A - CUBE RESULTS

### Step 2: Fix the column types

Dates come in as text, so we change them to real dates. Then we add year and month for the trends later.

In [2]:
cubes["date_cast"] = pd.to_datetime(cubes["date_cast"], errors="coerce")
cubes["year"] = cubes["date_cast"].dt.year
cubes["month"] = cubes["date_cast"].dt.to_period("M").astype(str)

print(cubes[["date_cast", "year", "month"]].head())

  date_cast  year month
0       NaT   NaN   NaT
1       NaT   NaN   NaT
2       NaT   NaN   NaT
3       NaT   NaN   NaT
4       NaT   NaN   NaT


### Step 3: Start the remove_reason column

Empty for now. Each rule below fills it in for the rows it catches.
We also keep a small log of how many rows each rule caught.

In [3]:
cubes["remove_reason"] = ""
log = []

def mark(rows, reason):
    # only mark rows that do not already have a reason, so each row is counted once
    new = rows & (cubes["remove_reason"] == "")
    cubes.loc[new, "remove_reason"] = reason
    log.append({"rule": reason, "rows": int(new.sum())})
    print(reason, "->", int(new.sum()), "rows")

### Step 4: Duplicates

The same cube set can appear twice (for example in the 7-day and the 28-day report).
A row is a duplicate if the pour, age and the three cube values are all the same.

In [4]:
same = ["lr", "element", "age", "cube1", "cube2", "cube3"]
duplicates = cubes.duplicated(subset=same, keep="first")
mark(duplicates, "duplicate")

duplicate -> 0 rows


### Step 5: Kerb mix

Kerbs are machine-made with a very dry mix (slump about 0 mm).
That is not the D2 mix, so the cement, fly ash and water columns are wrong for these rows.
We take them out of the main analysis and can look at them on their own later.

In [5]:
kerb = cubes["folder"].str.contains("Kerbs", na=False) & (cubes["measured_slump"] < 25)
mark(kerb, "kerb mix (dry, not the D2 mix)")

kerb mix (dry, not the D2 mix) -> 516 rows


### Step 6: Impossible strengths

Concrete on this job does not go above about 90 MPa.
A cube of 150 or 272 MPa is a typing or reading error on the report.
We also take out anything under 5 MPa - that is not a real result for structural concrete.

In [6]:
highest_cube = cubes[["cube1", "cube2", "cube3"]].max(axis=1)

too_high = (cubes["strength"] > 100) | (highest_cube > 100)
mark(too_high, "strength above 100 MPa (error)")

too_low = cubes["strength"] < 5
mark(too_low, "strength below 5 MPa (error)")

strength above 100 MPa (error) -> 1 rows
strength below 5 MPa (error) -> 0 rows


### Step 7: Cubes in a set that do not agree

The three cubes in a set come from the same concrete, so they should be close.
If the highest and lowest cube differ by more than 30% of the average, something went wrong with the cubes
(making, curing or testing), not with the concrete.

In [7]:
lowest_cube = cubes[["cube1", "cube2", "cube3"]].min(axis=1)
cubes["cube_spread_percent"] = ((highest_cube - lowest_cube) / cubes["strength"] * 100).round(1)

big_spread = cubes["cube_spread_percent"] > 30
mark(big_spread, "cubes in set differ by more than 30%")

print(cubes["cube_spread_percent"].describe().round(1))

cubes in set differ by more than 30% -> 2 rows
count    4079.0
mean        5.5
std         4.9
min         0.0
25%         2.9
50%         4.8
75%         7.2
max       222.0
Name: cube_spread_percent, dtype: float64


### Step 8: Impossible densities

Normal concrete cubes are about 2,300 - 2,500 kg/m3.
Below 2,100 or above 2,600 is a badly made cube or a wrong reading.

In [8]:
bad_density = (cubes["density"] < 2100) | (cubes["density"] > 2600)
mark(bad_density, "density outside 2100-2600 kg/m3")

density outside 2100-2600 kg/m3 -> 0 rows


### Step 9: Group the ages

Most cubes are tested at 7 or 28 days. A few are tested a day late (8 or 29 days) - we count those with 7 and 28.
Everything else is a special test. We keep them, but label them "other" so we do not mix them in with the 28-day results.

In [9]:
def age_group(age):
    if age in (7, 8):
        return "7 days"
    if age in (28, 29):
        return "28 days"
    return "other"

cubes["age_group"] = cubes["age"].apply(age_group)
print(cubes["age_group"].value_counts())

age_group
7 days     2112
28 days    1900
other        67
Name: count, dtype: int64


### Step 10: Add some useful columns

- `binder` = cement + fly ash
- `water_binder` = water / binder (lower usually means stronger concrete)
- `fly_ash_percent` = how much of the binder is fly ash
- `strength_class` = the specified strength (30 for 30/20, 40 for W40/20 ...)
- `strength_vs_class` = strength / class (1.0 means exactly the class, 1.3 means 30% above)
- `passed` = 28-day result at or above the class

In [10]:
cubes["binder"] = cubes["cement"] + cubes["fly_ash"]
cubes["water_binder"] = (cubes["water"] / cubes["binder"]).round(3)
cubes["fly_ash_percent"] = (cubes["fly_ash"] / cubes["binder"] * 100).round(1)

# AFR-W40/20 -> 40, AFR-35/28CRCP -> 35
cubes["strength_class"] = cubes["mix_id"].str.extract(r"AFR-W?(\d+)").astype(int)
cubes["strength_vs_class"] = (cubes["strength"] / cubes["strength_class"]).round(2)

cubes["passed"] = None
day28 = cubes["age_group"] == "28 days"
cubes.loc[day28, "passed"] = cubes.loc[day28, "strength"] >= cubes.loc[day28, "strength_class"]

print(cubes[["mix_id", "binder", "water_binder", "fly_ash_percent", "strength_class", "strength", "strength_vs_class"]].head())

       mix_id  binder  water_binder  fly_ash_percent  strength_class  \
0   AFR-30/20   279.0         0.602             30.1              30   
1   AFR-30/20   279.0         0.602             30.1              30   
2  AFR-W50/20   427.0         0.417             30.0              50   
3  AFR-W50/20   427.0         0.417             30.0              50   
4   AFR-15/20   171.0         1.053             29.8              15   

   strength  strength_vs_class  
0      35.5               1.18  
1      50.8               1.69  
2      49.9               1.00  
3      65.9               1.32  
4      35.2               2.35  


### Step 11: Rows we keep but must be careful with

These are NOT removed, but we need to remember them:
- No cast date -> cannot be used for trends over time
- "class only" match -> the mix design is our best guess

In [11]:
print("No cast date:", cubes["date_cast"].isna().sum())
print("No measured slump:", cubes["measured_slump"].isna().sum())
print()
print(cubes["match"].value_counts())

No cast date: 73
No measured slump: 58

match
exact                      3081
class only                  821
element (CRCP pavement)     173
description (25MPa/20)        4
Name: count, dtype: int64


### Step 12: Summary of the cleaning

In [12]:
log_table = pd.DataFrame(log)
print(log_table)
print()

keep = cubes["remove_reason"] == ""
print("Rows to start with:", len(cubes))
print("Rows removed:", (~keep).sum())
print("Rows kept:", keep.sum())

                                   rule  rows
0                             duplicate     0
1        kerb mix (dry, not the D2 mix)   516
2        strength above 100 MPa (error)     1
3          strength below 5 MPa (error)     0
4  cubes in set differ by more than 30%     2
5       density outside 2100-2600 kg/m3     0

Rows to start with: 4079
Rows removed: 519
Rows kept: 3560


Results per mix after cleaning - check that every mix still has enough rows.

In [13]:
clean = cubes[keep].copy()
print(clean.groupby("mix_id")["age_group"].value_counts().unstack().fillna(0).astype(int))

age_group      28 days  7 days  other
mix_id                               
AFR-15/20           46      46      7
AFR-20/20          231     257      5
AFR-25/20          144     167      8
AFR-30/20          377     395      4
AFR-35/20           24      30      0
AFR-35/28CRCP       74      99      0
AFR-40/20          199     220      2
AFR-W30/20          37      42      0
AFR-W40/20         365     417      7
AFR-W50/14          28      32      4
AFR-W50/20         110     119     29
AFR-W50/20HS        17      17      1


## PART B - DURABILITY RESULTS

### Step 13: Fix the durability data

- Dates to real dates
- The copied chloride values are already in their own column (`chloride_reported_suspect`), so we just do not use it
- LR-3633 / 3634 has a negative sorptivity disk, which is impossible, so that mean sorptivity is not valid

In [14]:
durability["date_cast"] = pd.to_datetime(durability["date_cast"], errors="coerce")
durability["year"] = durability["date_cast"].dt.year

sorp_columns = ["sorp_1", "sorp_2", "sorp_3", "sorp_4"]
negative = (durability[sorp_columns] < 0).any(axis=1)
print("Tests with a negative sorptivity disk:", negative.sum())
print(durability.loc[negative, ["lr", "sorp_1", "sorp_2", "sorp_3", "sorp_4", "sorptivity_mean"]])

durability.loc[negative, "sorptivity_mean"] = None

Tests with a negative sorptivity disk: 2
          lr  sorp_1  sorp_2  sorp_3  sorp_4  sorptivity_mean
156  LR-3633     6.9  -0.268    6.78     9.4              5.7
157  LR-3634     6.9  -0.268    6.78     9.4              5.7


### Step 14: Tests that share one panel

Sometimes one test panel is reported for several pours (same panel number, same results).
We keep every row (each is a different pour), but count how many rows share the panel
so we do not think we have more independent tests than we really have.

In [15]:
durability["rows_sharing_panel"] = durability.groupby(["panel", "opi_mean"])["lr"].transform("count")

shared = durability[durability["rows_sharing_panel"] > 1]
print("Rows that share a panel with another row:", len(shared))
print(shared[["lr", "panel", "element", "opi_mean", "rows_sharing_panel"]].head(10))

Rows that share a panel with another row: 34
          lr   panel                                            element  \
68   LR-1247  1230.0                    Ramp A - Parapet 3 Western Side   
69   LR-1287  1230.0                                 Ramp A - Parapet 1   
112  LR-3063  1547.0  Ramp B - Deck Parapet Type A - Parapet 4 on So...   
113  LR-3064  1547.0  Ramp A - MSE Wall Parapet - Type A5 - Parapet ...   
114  LR-3065  1547.0  Ramp A - MSE Wall Parapet - Type A5 - Parapet ...   
115  LR-3070  1547.0  Ramp C - Deck Parapet - Type A - Parapet 4 on ...   
116  LR-3071  1547.0  Ramp C - Deck Parapet - Type A - Parapet 4 on ...   
120  LR-3108  1550.0  Ramp D- Retaining wall- South side panel 2 Ret...   
121  LR-3109  1550.0  Ramp D - Retaining Wall - South Side Panel 4 R...   
122  LR-3110  1550.0  Ramp D - Retaining Wall - North Side Panel 4 R...   

     opi_mean  rows_sharing_panel  
68      10.17                 2.0  
69      10.17                 2.0  
112      9.45        

## PART C - MIX DESIGNS

### Step 15: Add the same useful columns to the mix designs

In [16]:
mixes["binder"] = mixes["cement"] + mixes["fly_ash"]
mixes["water_binder"] = (mixes["water"] / mixes["binder"]).round(3)
mixes["fly_ash_percent"] = (mixes["fly_ash"] / mixes["binder"] * 100).round(1)
mixes["strength_class"] = mixes["mix_id"].str.extract(r"AFR-W?(\d+)").astype(int)
print(mixes[["mix_id", "strength_class", "binder", "water_binder", "fly_ash_percent"]])

           mix_id  strength_class  binder  water_binder  fly_ash_percent
0       AFR-15/20              15   171.0         1.053             29.8
1       AFR-20/20              20   217.0         0.820             30.0
2       AFR-25/20              25   254.0         0.669             29.9
3       AFR-30/20              30   279.0         0.602             30.1
4       AFR-35/20              35   289.0         0.581             30.1
5       AFR-40/20              40   341.0         0.490             29.9
6      AFR-W30/20              30   337.0         0.549             30.0
7      AFR-W40/20              40   406.0         0.438             30.0
8      AFR-W50/20              50   427.0         0.417             30.0
9      AFR-W50/14              50   434.0         0.410             30.0
10   AFR-W50/20HS              50   432.0         0.412             30.1
11  AFR-35/28CRCP              35   320.0         0.531              0.0


## PART D - SAVE

### Step 16: Save the files

In [17]:
interim = project / "data" / "interim"
processed = project / "data" / "processed"
reports = project / "reports"

cubes.to_csv(interim / "cubes_with_reasons.csv", index=False)
clean.to_csv(processed / "cubes_clean.csv", index=False)
durability.to_csv(processed / "durability_clean.csv", index=False)
mixes.to_csv(processed / "mix_designs.csv", index=False)
log_table.to_csv(reports / "cleaning_log.csv", index=False)

print("Saved:")
print(" cubes_with_reasons.csv", cubes.shape)
print(" cubes_clean.csv", clean.shape)
print(" durability_clean.csv", durability.shape)
print(" mix_designs.csv", mixes.shape)

Saved:
 cubes_with_reasons.csv (4079, 42)
 cubes_clean.csv (3560, 42)
 durability_clean.csv (198, 31)
 mix_designs.csv (12, 20)


### Step 17: Write down what you decided

Answer these in your own words before Step 1c:
1. How many rows did we start with, and how many are left?
2. Which rule removed the most rows, and do you agree with it?
3. Is there any rule you would change (for example the 30% cube spread or the 5 MPa limit)?
4. Which mixes now have very few results, and what does that mean for our conclusions?